# Mario DRL — Notebook de Análise (interativo)

Este notebook é para **explorar os dados** já gerados pelos scripts CLI:

- `python scripts/train.py --profile full`  → gera os modelos e logs
- `python scripts/analyze.py`               → gera CSVs e PNGs base

Aqui você pode iterar sobre tabelas, filtrar fases/algoritmos, plotar curvas
customizadas, e gerar comparações visuais sob demanda. Tudo importa de `src/`,
então qualquer mudança nos módulos se propaga sem refatorar o notebook.

## 1. Imports e setup

In [ ]:
import sys
from pathlib import Path

# Garante que src/ é importável
PROJECT_ROOT = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display

from src import config
from src.analysis import (
    load_all_logs, run_full_analysis,
    compute_group1_metrics, aggregate_group1,
    compute_group2_metrics, aggregate_group2,
    spearman_difficulty, mannwhitney_between_algos,
)
from src.plots import (
    plot_learning_curves, plot_group1_comparison, plot_group2_difficulty,
)

sns.set_theme(style="whitegrid", context="paper")
pd.set_option("display.max_rows", 50)

print(f"Lendo de: {config.ROOT_DIR}")

## 2. Carregamento dos logs

In [ ]:
df_all = load_all_logs()
print(f"Avaliações: {len(df_all):,}")
if not df_all.empty:
    print(f"Configurações: {df_all.groupby(['algo','stage','seed']).ngroups}")
    print(f"Algoritmos: {sorted(df_all['algo'].unique())}")
    print(f"Fases    : {sorted(df_all['stage'].unique())}")
    print(f"Seeds    : {sorted(df_all['seed'].unique())}")
    display(df_all.head())

## 3. Métricas Grupo I — comparação entre arquiteturas

- $\bar{R}_{\text{final}}$: média de recompensa nos últimos 50k timesteps
- AUC normalizada da curva de aprendizado
- $t_{80\%}$: timesteps até atingir 80% do máximo
- $\sigma_{\text{final}}$: desvio-padrão dos últimos 50k

In [ ]:
metrics_g1 = compute_group1_metrics(df_all)
metrics_g1_agg = aggregate_group1(metrics_g1)
print("=== Grupo I — agregado por (algo, stage) ===\n")
display(metrics_g1_agg.round(2))

## 4. Métricas Grupo II — avaliação de dificuldade

- $\tau$: taxa de conclusão
- $\bar{d}$: distância normalizada (max_x / comprimento da fase)
- mortes/episódio, tempo médio

In [ ]:
metrics_g2 = compute_group2_metrics(df_all)
metrics_g2_agg = aggregate_group2(metrics_g2)
print("=== Grupo II — agregado por (algo, stage) ===\n")
display(metrics_g2_agg.round(3))

## 5. Análise estatística — Spearman

In [ ]:
spearman_df = spearman_difficulty(metrics_g2_agg)
print("=== Spearman: ranking do agente vs dificuldade canônica ===\n")
display(spearman_df.round(3))

## 6. Mann-Whitney U entre algoritmos

In [ ]:
mw_df = mannwhitney_between_algos(metrics_g1, metric="R_final")
print("=== Mann-Whitney U (R_final, por fase) ===\n")
display(mw_df.round(4))

## 7. Plots — curvas de aprendizado

In [ ]:
plot_learning_curves(df_all)

## 8. Plots — comparação Grupo I

In [ ]:
plot_group1_comparison(metrics_g1)

## 9. Plots — Grupo II vs dificuldade canônica

In [ ]:
plot_group2_difficulty(metrics_g2)

## 10. Visualização animada inline — GIF de agente

Para visualizar GIFs no notebook (animação real, não estática), usamos
matplotlib FuncAnimation com to_jshtml().

In [ ]:
import matplotlib.animation as _mpl_animation
from src.visualization import run_episode_for_render
from src.utils import get_device
from stable_baselines3 import DQN, PPO, A2C


def animate_frames_inline(frames, fps=15, figsize=(6, 5)):
    """Anima inline via FuncAnimation.to_jshtml — funciona em qualquer cliente."""
    if not frames:
        return HTML("<i>Nenhum frame.</i>")
    fig, ax = plt.subplots(figsize=figsize); ax.axis("off")
    im = ax.imshow(frames[0])
    def update(i):
        im.set_array(frames[i])
        return [im]
    ani = _mpl_animation.FuncAnimation(fig, update, frames=len(frames),
                                       interval=1000.0/fps, blit=True)
    html = ani.to_jshtml(default_mode="loop")
    plt.close(fig)
    return HTML(html)


# CONFIGURE aqui qual modelo visualizar
ALGO_TO_VIEW  = "PPO"
STAGE_TO_VIEW = "1-1"
SEED_TO_VIEW  = 42

_ALGO_CLS = {"DQN": DQN, "PPO": PPO, "A2C": A2C}
model_path = config.MODELS_DIR / f"{ALGO_TO_VIEW}_stage{STAGE_TO_VIEW}_seed{SEED_TO_VIEW}.zip"

if model_path.exists():
    model = _ALGO_CLS[ALGO_TO_VIEW].load(model_path, device=get_device())
    frames, metrics = run_episode_for_render(model, stage=STAGE_TO_VIEW, max_steps=2000, seed=999)
    print(f"reward={metrics['reward']:+.1f}  max_x={metrics['max_x']}  "
          f"flag={metrics['flag_get']}  deaths={metrics['deaths']}")
    display(animate_frames_inline(frames, fps=15))
else:
    print(f"Modelo não encontrado: {model_path.name}")

## 11. Comparação DQN vs PPO vs A2C lado-a-lado (animado)

In [ ]:
from src.visualization import _ALGO_CLS as _CLS, render_models_side_by_side

stage_for_render = "1-1"
seed_for_render = 42

models = {}
for algo in config.ALGOS:
    path = config.MODELS_DIR / f"{algo}_stage{stage_for_render}_seed{seed_for_render}.zip"
    if path.exists():
        models[algo] = _CLS[algo].load(path, device=get_device())
        print(f"✓ {algo}: {path.name}")
    else:
        print(f"✗ {algo} não encontrado: {path.name}")

if len(models) >= 2:
    _, all_metrics = render_models_side_by_side(
        models, stage=stage_for_render, max_steps=3000, render_seed=999,
    )
    # Re-renderiza os mesmos episódios para animação inline
    all_frames = {}
    for label, m in models.items():
        frames, _ = run_episode_for_render(m, stage_for_render, 3000, 999)
        all_frames[label] = frames

    max_len = max(len(f) for f in all_frames.values())
    composed = []
    for t in range(max_len):
        panels = []
        for label in all_frames:
            f = all_frames[label]
            panels.append(f[t] if t < len(f) else f[-1])
        composed.append(np.concatenate(panels, axis=1))
    display(animate_frames_inline(composed, fps=15, figsize=(12, 4)))

## 12. Resumo de artefatos

Tudo em `mario_drl_results/`:

```
mario_drl_results/
├── models/                          # *.zip — modelos finais
│   └── checkpoints/                 # *_{N}_steps.zip — para evolução temporal
├── logs/                            # *.csv — métricas por episódio
├── tensorboard/                     # logs do TensorBoard
├── metrics/                         # CSVs consolidados
│   ├── group1_per_seed.csv
│   ├── group1_aggregated.csv
│   ├── group2_per_seed.csv
│   ├── group2_aggregated.csv
│   ├── spearman_difficulty.csv
│   └── mannwhitney_algos.csv
└── plots/                           # PNGs + GIFs (curvas, comparações, evolução)
```

TensorBoard ao vivo: `tensorboard --logdir mario_drl_results/tensorboard`